# Gaussian network model

Read a local structure and choose the explicit NumPy engine. The node selection
defaults to alpha carbons; the default contact cutoff is 7 angstroms.

In [ ]:
from importlib.resources import as_file, files
import numpy as np
import molsysmt as msm
from elastnetmt import AnisotropicNetworkModel, GaussianNetworkModel, pyunitwizard as puw

resource = files("molsysmt").joinpath("data/pdb/1tcd.pdb")
with as_file(resource) as path:
    molecular_system = msm.convert(str(path), to_form="molsysmt.MolSys")

In [ ]:
gnm = GaussianNetworkModel(molecular_system, engine="vectorized")
assert gnm.contacts.shape == (gnm.n_nodes, gnm.n_nodes)
assert not gnm.contacts.diagonal().any()
values = gnm.get_eigenvalues()
assert values.shape == (gnm.n_nodes,)
assert values[0] == 0 and np.all(values[1:] > 0)

## Prediction and calibration

`get_b_factors(n_modes="all")` initially returns a dimensionless raw profile.
A positive integer selects that many lowest non-rigid modes. After a complete
experimental fit, the same fitted scale applies to truncated predictions.

In [ ]:
raw = gnm.get_b_factors()
scale, correlation = gnm.fit_to_experimental_b_factors()
np.testing.assert_allclose(gnm.get_b_factors(), raw * scale)
partial = gnm.get_b_factors(n_modes=10)
assert partial.shape == raw.shape
assert np.all(partial <= gnm.get_b_factors() + 1e-10)

In [ ]:
gnm.show_b_factors()
gnm.show_contact_map()

## Recalculation

Changing contacts invalidates spectra and experimental calibration. The next
prediction is raw again; repeat the fit when calibrated output is needed.

In [ ]:
gnm.calculate_contacts(cutoff="8 angstroms")
assert gnm.scaling_factor == 1.0 and gnm.b_factors_exp is None
assert gnm.kirchhoff_matrix is None
assert np.all(np.isfinite(gnm.get_b_factors()))

Absent experimental observations permit raw predictions. Missing/invalid
experimental data raise `ArgumentError` during fitting; constant profiles
raise `UndefinedCorrelationError`. See the [API](../../../../api/users/models.rst).